# GFS 2 m temperature — NWP quickstart

Fetch a recent GFS run's 2 m temperature over a small bbox and read the resulting Cloud-Optimized GeoTIFF.

**Requirements** (live download):

```bash
pip install earthlens[nwp]
conda install -c conda-forge eccodes libgdal-grib   # binary libs
```

`herbie-data` downloads only the `.idx` byte range for the requested variable (>99 % bandwidth saving); `pyramids.grib.open_grib` reads it with GDAL's GRIB driver, and earthlens crops it to the bbox.

In [1]:
import datetime as dt

from earthlens.earthlens import EarthLens

# A recent 00Z cycle (GFS keeps ~10 days on NODD).
cycle_day = (dt.datetime.now(dt.UTC) - dt.timedelta(days=1)).strftime('%Y-%m-%d')
cycle_day

'2026-05-26'

In [2]:
lens = EarthLens(
    data_source='nwp',
    dataset='gfs',
    variables=['temperature_2m'],
    start=cycle_day,
    end=cycle_day,
    aoi=[-80, 40, -75, 45],
    path='out/gfs',
    steps=[0],
    mirror='aws',
)
paths = lens.download(progress_bar=False)
[p.name for p in paths]

2026-05-27 15:51:39 | INFO | pyramids.base.config | Logging is configured.


2026-05-27 15:51:40 | INFO | herbie.core | `product` not specified. Will use "pgrb2.0p25".


2026-05-27 15:51:41 | INFO | herbie.core | `product` not specified. Will use "pgrb2.0p25".


2026-05-27 15:51:42 | INFO | herbie.core | `product` not specified. Will use "pgrb2.0p25".


2026-05-27 15:51:43 | INFO | herbie.core | `product` not specified. Will use "pgrb2.0p25".


['gfs_2026052600_f000.tif',
 'gfs_2026052606_f000.tif',
 'gfs_2026052612_f000.tif',
 'gfs_2026052618_f000.tif']

In [3]:
from pyramids.grib import open_grib

# The output is a standard COG - open it with pyramids and inspect the grid.
ds = open_grib(str(paths[0]))
ds.shape

(1, 19, 19)

To request several lead times, pass `steps=[0, 6, 12, 24]` (or `horizon=48`); each `(cycle, step)` yields one COG. Add `aggregate=AggregationConfig(freq="1D", op="mean")` to reduce the stack to daily means.